# ProxyLens 02: base vs fine-tuned predictions on GPU

**What:** runs the base `Qwen2.5-3B-Instruct` (zero-shot, same prompt) and the fine-tuned LoRA on
an eval split and saves raw outputs. Metrics are **not** computed here: the repo's eval harness
(`eval/`, Phase 7) scores every provider the same way, and the gold labels never leave the repo.

**Inputs:** `<you>/proxylens-sft-v1` (`eval_inputs/test.jsonl` = gold-split prompts without labels,
`eval_inputs/val.jsonl`); the LoRA from notebook 01; Colab secret `HF_TOKEN`.

**Outputs:** `eval/results/<run_id>.json` per model (format in `eval/README.md`), uploaded to the
dataset repo under `results/` and offered as a download.

**Runtime:** free T4, roughly 10-25 s per item per model; ~184 test items x 2 models ≈ 1.5-2.5 h.
Each model's results are saved as soon as it finishes.

In [ ]:
%%capture
# Unsloth pulls matching torch / transformers / trl / peft / bitsandbytes for Colab.
!pip install -q unsloth
!pip install -q "huggingface_hub>=0.25" "pydantic>=2.8"

In [ ]:
from unsloth import FastLanguageModel  # import first: Unsloth patches transformers/trl

import glob, json, os, time
import torch
from google.colab import drive, userdata
from huggingface_hub import HfApi, snapshot_download

print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

HF_TOKEN = userdata.get("HF_TOKEN")  # Colab secret; needs write access
HF_USER = HfApi(token=HF_TOKEN).whoami()["name"]
DATASET_REPO = f"{HF_USER}/proxylens-sft-v1"           # pushed by scripts/push_sft_to_hub.py
LORA_REPO = f"{HF_USER}/proxylens-qwen3b-lora-v1"
GGUF_REPO = f"{HF_USER}/proxylens-qwen3b-gguf-v1"
GGUF_FILE = "proxylens-q4_k_m.gguf"                     # = STUDENT_GGUF_FILE in .env
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
MAX_SEQ_LEN = 4096        # longest SFT example is ~4.1k tokens
MAX_NEW_TOKENS = 1024     # longest teacher answer is ~870 tokens
SEED = 3407
print("HF user:", HF_USER)

In [ ]:
SPLIT = "test"   # "test" = gold-split items (score after hand-labelling); "val" = teacher-labelled
LIMIT = None     # e.g. 20 for a quick run
inputs_dir = snapshot_download(DATASET_REPO, repo_type="dataset", token=HF_TOKEN,
                               allow_patterns=["eval_inputs/*"])
rows = [json.loads(line) for line in open(f"{inputs_dir}/eval_inputs/{SPLIT}.jsonl")]
rows = rows[:LIMIT] if LIMIT else rows
print(len(rows), "items from", SPLIT)

In [ ]:
def generate(model, messages, max_new_tokens=MAX_NEW_TOKENS):
    """Greedy decode for one chat (system + user); returns the reply and its latency."""
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    started = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    latency_ms = int((time.perf_counter() - started) * 1000)
    return tokenizer.decode(new_tokens, skip_special_tokens=True), latency_ms, len(new_tokens)

In [ ]:
import datetime, gc

def run(provider, model_id):
    global tokenizer
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_id, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True, dtype=None, token=HF_TOKEN
    )
    FastLanguageModel.for_inference(model)
    items = []
    for i, row in enumerate(rows):
        text, latency_ms, n_tokens = generate(model, row["messages"])
        items.append({"item_id": row["item_id"], "output_text": text,
                      "latency_ms": latency_ms, "output_tokens": n_tokens})
        if i % 20 == 0:
            print(provider, i, latency_ms, "ms")
    stamp = datetime.datetime.now(datetime.timezone.utc)
    result = {
        "run_id": f"{stamp:%Y-%m-%d}_{provider}_{SPLIT}",
        "provider": provider,
        "model": model_id,
        "split": SPLIT,
        "created_at": stamp.isoformat(),
        "hardware": torch.cuda.get_device_name(0),
        "decoding": {"greedy": True, "max_new_tokens": MAX_NEW_TOKENS, "constrained": False},
        "items": items,
    }
    os.makedirs("results", exist_ok=True)
    path = f"results/{result['run_id']}.json"
    with open(path, "w") as fh:
        json.dump(result, fh)
    HfApi(token=HF_TOKEN).upload_file(path_or_fileobj=path, path_in_repo=path,
                                      repo_id=DATASET_REPO, repo_type="dataset")
    del model
    gc.collect(); torch.cuda.empty_cache()
    return path

In [ ]:
paths = [
    run("base_hf", BASE_MODEL),
    run("finetuned_lora", LORA_REPO),
]
print(paths)

In [ ]:
from google.colab import files
for path in paths:
    files.download(path)   # copy into the repo's eval/results/